In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:

import sys

PROJECT_ROOT = (
    "/Workspace/Users/anmolmishra02@gmail.com/"
    "NYC-Trip-Records-Pipeline/take_two_nyc_taxi"
)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)


from src.silver.transformations import (
    standardize_taxi_data,
    build_taxi_zone_dimension
)

from src.silver.data_quality import (
    apply_data_quality_checks
)


yellow_bronze = spark.table(
    "workspace.nyc_taxi_bronze.yellow_trips"
)

green_bronze = spark.table(
    "workspace.nyc_taxi_bronze.green_trips"
)

zone_bronze = spark.table(
    "workspace.nyc_taxi_bronze.taxi_zone_lookup"
)

In [0]:
yellow_silver = standardize_taxi_data(
    yellow_bronze,
    "yellow"
)

green_silver = standardize_taxi_data(
    green_bronze,
    "green"
)
unified_df = yellow_silver.unionByName(
    green_silver
)

In [0]:
valid_df, quarantine_df = apply_data_quality_checks(
    unified_df
)

zone_dimension_df = build_taxi_zone_dimension(
    zone_bronze
)

In [0]:
import importlib
import src.silver.transformations as transformations

importlib.reload(transformations)
importlib.reload(quality)

standardize_taxi_data = transformations.standardize_taxi_data
build_taxi_zone_dimension = transformations.build_taxi_zone_dimension

yellow_silver = standardize_taxi_data(
    yellow_bronze,
    "yellow"
)

green_silver = standardize_taxi_data(
    green_bronze,
    "green"
)

print("Yellow columns:", yellow_silver.columns)
print("Green columns:", green_silver.columns)

print("Yellow count:", len(yellow_silver.columns))
print("Green count:", len(green_silver.columns))

unified_df = yellow_silver.unionByName(green_silver)

In [0]:
%sql
ALTER TABLE workspace.nyc_taxi_silver.unified_trips
SET TBLPROPERTIES (
    'delta.feature.timestampNtz' = 'supported'
);
ALTER TABLE workspace.nyc_taxi_silver.quarantined_trips
SET TBLPROPERTIES (
    'delta.feature.timestampNtz' = 'supported'
);

In [0]:
SILVER_SCHEMA = "workspace.nyc_taxi_silver"


def write_silver_table(df, table_name):

    full_table_name = f"{SILVER_SCHEMA}.{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )

    print(f"Written successfully: {full_table_name}")


write_silver_table(valid_df, "unified_trips")

write_silver_table(quarantine_df, "quarantined_trips")

write_silver_table(zone_dimension_df, "dim_taxi_zones")

In [0]:
%sql
SELECT
    (SELECT COUNT(*) FROM workspace.nyc_taxi_silver.unified_trips)
    +
    (SELECT COUNT(*) FROM workspace.nyc_taxi_silver.quarantined_trips)
    AS silver_total_records;

In [0]:
%sql
SELECT
    dq_failure_reason,
    COUNT(*) AS records
FROM workspace.nyc_taxi_silver.quarantined_trips
GROUP BY dq_failure_reason;

In [0]:
%sql
SELECT
    service_type,
    source_month,
    pickup_datetime,
    dropoff_datetime,
    TIMESTAMPDIFF(
        SECOND,
        dropoff_datetime,
        pickup_datetime
    ) AS negative_duration_seconds
FROM workspace.nyc_taxi_silver.quarantined_trips
LIMIT 20;

In [0]:
%sql
SELECT
    service_type,
    COUNT(*) AS records
FROM workspace.nyc_taxi_silver.unified_trips
GROUP BY service_type;

In [0]:
print("Current Spark timezone:",
      spark.conf.get("spark.sql.session.timeZone"))

yellow_path = (
    "/Volumes/workspace/nyc_taxi/source_data/Yellow/"
    "yellow_tripdata_2025-01.parquet"
)

green_path = (
    "/Volumes/workspace/nyc_taxi/source_data/green/"
    "green_tripdata_2025-01.parquet"
)

yellow_df = spark.read.parquet(yellow_path)
green_df = spark.read.parquet(green_path)

print("YELLOW SCHEMA")
yellow_df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
).printSchema()

print("GREEN SCHEMA")
green_df.select(
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime"
).printSchema()
yellow_df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
).show(10, truncate=False)

green_df.select(
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime"
).show(10, truncate=False)

In [0]:
from pyspark.sql import functions as F

for service, df, pickup_col, dropoff_col in [
    ("yellow", yellow_df, "tpep_pickup_datetime", "tpep_dropoff_datetime"),
    ("green", green_df, "lpep_pickup_datetime", "lpep_dropoff_datetime")
]:
    print(f"--- {service.upper()} ---")

    df.select(
        F.min(pickup_col).alias("min_pickup"),
        F.max(pickup_col).alias("max_pickup"),
        F.min(dropoff_col).alias("min_dropoff"),
        F.max(dropoff_col).alias("max_dropoff")
    ).show(truncate=False)

In [0]:
%sql
WITH all_trips AS (
    SELECT
        source_month,
        'yellow' AS service_type,
        source_file,
        tpep_pickup_datetime AS pickup_datetime,
        tpep_dropoff_datetime AS dropoff_datetime
    FROM workspace.nyc_taxi_bronze.yellow_trips

    UNION ALL

    SELECT
        source_month,
        'green' AS service_type,
        source_file,
        lpep_pickup_datetime,
        lpep_dropoff_datetime
    FROM workspace.nyc_taxi_bronze.green_trips
),

windows AS (
    SELECT *,
        TO_DATE(CONCAT(source_month, '-01')) AS month_start
    FROM all_trips
)

SELECT *
FROM windows
WHERE
    TO_DATE(pickup_datetime) < DATE_SUB(month_start, 1)
    OR TO_DATE(pickup_datetime) >= DATE_ADD(ADD_MONTHS(month_start, 1), 1)
    OR TO_DATE(dropoff_datetime) < DATE_SUB(month_start, 1)
    OR TO_DATE(dropoff_datetime) >= DATE_ADD(ADD_MONTHS(month_start, 1), 1)
ORDER BY source_month, service_type, pickup_datetime;

In [0]:
%sql
SELECT
    dq_warning_reason,
    COUNT(*) AS record_count
FROM workspace.nyc_taxi_silver.unified_trips
WHERE cbd_congestion_fee > 0
  AND dropoff_datetime <
      CAST('2025-01-05 00:00:00' AS TIMESTAMP_NTZ)
GROUP BY dq_warning_reason;